<a href="https://colab.research.google.com/github/dhiyasalmas/Parallel-Data-Processing/blob/main/UTS%20Dhiya%20Salma%20S.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UTS Jaringan dan Pengelolaan Data Paralel

Selesaikan integral berikut dengan menggunakan metode numerik trapezoid dengan cara serial, reduce, multiprocessing, multithreading, dan CUDA. Bandingkan hasil dan speedupnya!

$$\int_{3}^{4} \int_{1}^{2} xy \, dx \, dy$$



## Serial

In [1]:
import time
start=time.time()

def f(x,y):
    return x*y

def uts(a, b, c, d, nx, ny):
    hx = (b - a) / nx
    hy = (d - c) / ny
    s = 0.0
    for i in range(nx):
    	for j in range(ny):
          x = a + i * hx
          y = c + j * hy
          s += f(x,y)
    return hy * hx * s

a = 1
b = 2
c = 3
d = 4
nx = 10**4
ny = 10**4
print("numerik =",uts(a,b,c,d,nx,ny))
print("waktu serial", time.time()-start)

numerik = 5.249750002499476
waktu serial 27.364213466644287


## Reduce

In [2]:
!sudo apt-get install openmpi-bin libopenmpi-dev
!pip install mpi4py

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libopenmpi-dev is already the newest version (4.1.2-2ubuntu1).
libopenmpi-dev set to manually installed.
openmpi-bin is already the newest version (4.1.2-2ubuntu1).
openmpi-bin set to manually installed.
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 48.8 MB/s eta 0:00:00


In [3]:
from mpi4py import MPI
import timeit
comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()
start = MPI.Wtime()
def f(x):
    return x

def f(y):
    return y

def utsx(a, b, nx):
    hy = (d - c) / ny
    s = 0.0
    for i in range(nx):
          x = a + i * hx
          s += f(x)
    return hx * s

def utsy(c, d, ny):
    hy = (d - c) / ny
    s = 0.0
    for j in range(ny):
          y = c + j * hy
          s += f(y)
    return hy * s

a = 1
b = 2
c = 3
d = 4
nx = 10**4
ny = 10**4
hx = (b - a)/nx
hy = (d - c)/ny

local_nx = nx // size
local_ny = ny // size
local_a = a + rank * local_nx * hx
local_b = local_a + local_nx * hx
local_c = c + rank * local_ny * hy
local_d = local_c + local_ny * hy
local_integralx = utsx(local_a, local_b, local_nx)
local_integraly = utsy(local_c, local_d, local_ny)

my_integralx=local_integralx
my_integraly = local_integraly
my_integralx=comm.reduce(my_integralx, op=MPI.SUM, root=0)
my_integraly=comm.reduce(my_integraly, op=MPI.SUM, root=0)
if rank == 0:
    print("hasil numerik = ", my_integralx*my_integraly)
    print("waktu = ", MPI.Wtime()-start, "detik")

hasil numerik =  5.2497500025
waktu =  0.003551543 detik


## Multiprocessing

In [5]:
from multiprocessing import Pool, cpu_count
import time

def f(x, y):
    return x * y

def uts_worker(args):
    a_sub, b_sub, nx_sub, c, d, ny = args
    hx = (b_sub - a_sub) / nx_sub
    hy = (d - c) / ny

    total = 0.0
    for i in range(nx_sub):
        for j in range(ny):
            x = a_sub + i * hx
            y = c + j * hy
            total += f(x, y)
    return total * hx * hy

if __name__ == "__main__":
    a, b, c, d = 1.0, 2.0, 3.0, 4.0
    nx, ny = 10**4, 10**4

    start = time.time()
    num_proc = cpu_count()
    chunk_size = nx // num_proc

    tasks = []
    for i in range(num_proc):
        start_x = a + i * (chunk_size * (b - a) / nx)
        end_x = start_x + (chunk_size * (b - a) / nx)
        tasks.append((start_x, end_x, chunk_size, c, d, ny))
    with Pool(processes=num_proc) as pool:
        results = pool.map(uts_worker, tasks)
    total_integral = sum(results)
    end = time.time()

    print(f"Hasil integral multiprocessing: {total_integral}")
    print(f"Waktu eksekusi: {end - start:.4f} detik")

Hasil integral multiprocessing: 5.249750002498933
Waktu eksekusi: 18.9805 detik


## Multithreading

In [8]:
from concurrent.futures import ThreadPoolExecutor
import os
import time

def f(x, y):
    return x * y

def uts(a_sub, b_sub, nx_sub, c, d, ny):
    hx = (b_sub - a_sub) / nx_sub
    hy = (d - c) / ny
    total = 0.0
    for i in range(nx_sub):
        for j in range(ny):
            x = a_sub + i * hx
            y = c + j * hy
            total += f(x, y)
    return total * hx * hy

if __name__ == "__main__":
    a, b, c, d = 1.0, 2.0, 3.0, 4.0
    nx = 10**4
    ny = 10**4

    start = time.time()
    n_workers = os.cpu_count()
    chunkx = nx // n_workers

    futures = []
    with ThreadPoolExecutor(max_workers=n_workers) as executor:
        for i in range(n_workers):
            a_i = a + i * (chunkx * (b - a) / nx)
            b_i = a_i + (chunkx * (b - a) / nx)
            futures.append(executor.submit(uts, a_i, b_i, chunkx, c, d, ny))
        results = [f.result() for f in futures]

    print("Hasil integral multithreading :", sum(results))
    print("Waktu :", time.time() - start)

Hasil integral multithreading : 5.249750002498933
Waktu : 24.125884771347046


## PyCUDA

In [1]:
!pip install pycuda

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 17.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 12.3 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2026.1-cp312-cp312-linux_x86_64.whl size=659447 sha256=d8e05a26e32436f43b55ffe197ba9b7d1d961f37b394a70279da53152201aabf
  Stored in directory: /root/.cache/pip/wheels/90/2a/71/75ec0cc316cc0ff494bfffa2935e02580129cb7f859a0cfd8f
Successfully built pycuda


In [7]:
import pycuda.autoinit
import pycuda.driver as cuda
import numpy as np
from pycuda.compiler import SourceModule
import time

kernel_code = """
__global__ void uts_kernel(double a, double hx, double c, double hy, double *d_sum, int nx, int ny) {

    int i = threadIdx.x + blockIdx.x * blockDim.x;
    int j = threadIdx.y + blockIdx.y * blockDim.y;

    if (i < nx && j < ny) {
        double x = a + i * hx;
        double y = c + j * hy;
        double f = x * y;
    }
}
"""

def uts_gpu(a, b, c, d, nx, ny):
    hx = (b - a) / nx
    hy = (d - c) / ny

    h_sum = np.zeros(1, dtype=np.float64)
    d_sum = cuda.mem_alloc(h_sum.nbytes)
    cuda.memcpy_htod(d_sum, h_sum)

    mod = SourceModule(kernel_code)
    uts_kernel = mod.get_function("uts_kernel")
    threads_per_block = (16, 16, 1)
    grid_x = int(np.ceil(nx / threads_per_block[0]))
    grid_y = int(np.ceil(ny / threads_per_block[1]))

    uts_kernel(
        np.float64(a), np.float64(hx), np.float64(c), np.float64(hy), d_sum, np.int32(nx), np.int32(ny),
        block=threads_per_block, grid=(grid_x, grid_y)
    )

    cuda.memcpy_dtoh(h_sum, d_sum)
    integral = h_sum[0] * hx * hy
    return integral

a, b = 1.0, 2.0
c, d = 3.0, 4.0
nx, ny = 10**4, 10**4

start = time.time()
result = uts_gpu(a, b, c, d, nx, ny)
end = time.time()

print(f"Integral = {result}")
print(f"Waktu eksekusi (GPU): {end - start:.4f} detik")

Integral = 5.249750002501148
Waktu eksekusi (GPU): 0.8456 detik
